In [2]:
import os, zipfile
from pathlib import Path
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Suppress warnings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("c:/Users/kumar/OneDrive/Desktop/5000_resume")
JD_PDF = BASE_DIR / "20_Job_Descriptions (2).pdf"
RESUMES_DIR = BASE_DIR / "resumes"
THRESHOLD  = 0.10  # Adjusted for TF-IDF (lower than neural embeddings)

# Verify files exist
if not JD_PDF.exists():
    raise FileNotFoundError(f"Job descriptions PDF not found: {JD_PDF}")
if not RESUMES_DIR.exists():
    raise FileNotFoundError(f"Resumes directory not found: {RESUMES_DIR}")

# -----------------------------
# Helpers
# -----------------------------
def read_pdf(path):
    """Extract text from a PDF file."""
    try:
        pdf_path = Path(path)
        if not pdf_path.exists():
            print(f"Error: File not found {path}")
            return ""
        text = " ".join(p.extract_text() or "" for p in PdfReader(path).pages)
        return text
    except Exception as e:
        print(f"Error reading {path}: {e}")
        return ""

def get_resumes(resumes_dir):
    """Return list of (filename, text) for resumes from directory."""
    resumes = []
    resumes_path = Path(resumes_dir)
    
    if not resumes_path.is_dir():
        raise ValueError(f"{resumes_dir} is not a valid directory")
    
    resume_files = sorted(resumes_path.glob("*.pdf"))
    print(f"Reading {len(resume_files)} resume PDFs...")
    
    for idx, pdf_file in enumerate(resume_files, 1):
        text = read_pdf(str(pdf_file))
        if text.strip():
            resumes.append((pdf_file.name, text))
            if idx % 5 == 0:
                print(f"  ✓ Processed {idx}/{len(resume_files)}")
    
    return resumes

def get_job_descriptions(jd_pdf):
    """Return list of (title, text) for job descriptions (1 per page)."""
    jd_list = []
    try:
        reader = PdfReader(str(jd_pdf))
        print(f"Reading {len(reader.pages)} job descriptions from PDF...")
        for page_num, page in enumerate(reader.pages, 1):
            txt = page.extract_text()
            if txt:
                title = txt.strip().split("\n")[0] if txt.strip() else f"Job Description {page_num}"
                jd_list.append((title, txt))
                if page_num % 5 == 0:
                    print(f"  ✓ Extracted {page_num}/{len(reader.pages)}")
        return jd_list
    except Exception as e:
        print(f"Error reading job descriptions PDF: {e}")
        return []

# ===========================
# LOAD DATA
# ===========================
print("\n" + "="*80)
print("RESUME MATCHING SYSTEM - TF-IDF Based")
print("="*80 + "\n")

resumes = get_resumes(RESUMES_DIR)
print(f"✓ Loaded {len(resumes)} resumes\n")

jd_list = get_job_descriptions(JD_PDF)
print(f"✓ Loaded {len(jd_list)} job descriptions\n")

if not resumes or not jd_list:
    print("ERROR: No resumes or job descriptions found. Check file paths.")
else:
    # ===========================
    # COMPUTE SIMILARITY SCORES
    # ===========================
    print("Computing TF-IDF similarity scores...")
    
    # Prepare all texts
    all_texts = [text for _, text in resumes] + [text for _, text in jd_list]
    
    # Compute TF-IDF vectorizer
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    # Split into resume and JD embeddings
    resume_tfidf = tfidf_matrix[:len(resumes)]
    jd_tfidf = tfidf_matrix[len(resumes):]
    
    # Compute cosine similarity scores
    cosine_scores = cosine_similarity(resume_tfidf, jd_tfidf)
    print("✓ Similarity scores computed\n")
    
    # ===========================
    # REPORT RESULTS
    # ===========================
    print("="*80)
    print("RESUME-JD MATCHING RESULTS")
    print(f"Threshold: {THRESHOLD} | Matches: Resume → Job Description")
    print("="*80)
    
    match_count = 0
    results_by_jd = {jd_title: [] for jd_title, _ in jd_list}
    
    for i, (res_name, _) in enumerate(resumes):
        print(f"\n{i+1:2d}. {res_name}")
        found_match = False
        
        # Get top 3 matches for this resume
        top_matches_idx = np.argsort(cosine_scores[i])[::-1][:3]
        
        for rank, j in enumerate(top_matches_idx, 1):
            score = float(cosine_scores[i][j])
            jd_title = jd_list[j][0]
            
            if score >= THRESHOLD:
                print(f"     ✓ MATCH #{rank}: '{jd_title[:55]}' → {score:.4f}")
                results_by_jd[jd_title].append((res_name, score))
                found_match = True
                match_count += 1
            elif rank == 1:
                # Always show the best match
                print(f"     • Top match: '{jd_title[:55]}' → {score:.4f}")
    
    # Summary by job description
    print("\n" + "="*80)
    print("MATCHES BY JOB DESCRIPTION")
    print("="*80)
    matched_jds = 0
    for jd_title, matches in results_by_jd.items():
        if matches:
            print(f"\n✓ {jd_title}")
            for res_name, score in sorted(matches, key=lambda x: x[1], reverse=True):
                print(f"    → {res_name} ({score:.4f})")
            matched_jds += 1
    
    if matched_jds == 0:
        print("\nNo matches found. Consider lowering the threshold.")
    
    print("\n" + "="*80)
    print(f"SUMMARY: {match_count} total matches | {matched_jds}/{len(jd_list)} job descriptions matched")
    print(f"Threshold: {THRESHOLD} | Method: TF-IDF Cosine Similarity")
    print("="*80)



RESUME MATCHING SYSTEM - TF-IDF Based

Reading 20 resume PDFs...
  ✓ Processed 5/20
  ✓ Processed 10/20
  ✓ Processed 15/20
  ✓ Processed 20/20
✓ Loaded 20 resumes

Reading 20 job descriptions from PDF...
  ✓ Extracted 5/20
  ✓ Extracted 10/20
  ✓ Extracted 15/20
  ✓ Extracted 20/20
✓ Loaded 20 job descriptions

Computing TF-IDF similarity scores...
✓ Similarity scores computed

RESUME-JD MATCHING RESULTS
Threshold: 0.1 | Matches: Resume → Job Description

 1. data_science_01_david_kim.pdf
     ✓ MATCH #1: 'Data Scientist' → 0.1224

 2. data_science_02_sarah_williams.pdf
     ✓ MATCH #1: 'Machine Learning Engineer' → 0.1731

 3. data_science_03_james_okafor.pdf
     • Top match: 'Data Analyst' → 0.0658

 4. data_science_04_lisa_park.pdf
     ✓ MATCH #1: 'Power BI Developer' → 0.1535

 5. finance_01_michael_brown.pdf
     • Top match: 'Data Analyst' → 0.0379

 6. finance_02_jennifer_lee.pdf
     • Top match: 'Cloud Engineer' → 0.0188

 7. finance_03_robert_garcia.pdf
     • Top match: 